# Deep Learning project

Team members:

* Rimsha Afzal,
* Nika Dariani,
* Irina Krylova, 255809

## Setup

Run this notebook from the project root. In Colab, upload or mount the project folder first, then set `PROJECT_ROOT` to that folder.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the project root, where the src/ folder exists.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from baseline_retrieval import run_baseline
from embedding_io import load_embeddings

## Paths

The subset query file below is the one compatible with the copied test embeddings. The larger subset query file uses a different index space, so it is not used here.

In [ ]:
QUERY_JSON = PROJECT_ROOT / "data/celeba_subset/queries/test_embedding_celeba_evaluation.json"
IMAGE_EMBEDDING_DIR = PROJECT_ROOT / "data/celeba_subset/embeddings/test"
TEXT_EMBEDDING_DIR = PROJECT_ROOT / "data/celeba_subset/embeddings"
CHECK_JSON = PROJECT_ROOT / "data/celeba_subset/checks/query_embedding_compatibility_check.json"
SUMMARY_CSV = PROJECT_ROOT / "outputs/baseline_run/summary.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_run/notebook_test_subset_alpha2.00_beta1.00"

for path in [QUERY_JSON, IMAGE_EMBEDDING_DIR, TEXT_EMBEDDING_DIR, CHECK_JSON]:
    if not path.exists():
        raise FileNotFoundError(path)

## Data Sanity Check

Before running retrieval, check that the query file, image embeddings, and text embeddings refer to the same subset.

In [ ]:
with QUERY_JSON.open("r", encoding="utf-8") as handle:
    query_entries = json.load(handle)

with CHECK_JSON.open("r", encoding="utf-8") as handle:
    compatibility_check = json.load(handle)

image_embeddings, image_ids = load_embeddings(IMAGE_EMBEDDING_DIR, "image_embeddings", "cpu")
text_embeddings, text_ids = load_embeddings(TEXT_EMBEDDING_DIR, "text_embeddings", "cpu")

query_instances = sum(len(entry["ground_truth"]) for entry in query_entries)
target_links = sum(
    len(targets)
    for entry in query_entries
    for targets in entry["ground_truth"].values()
)

print("compatibility check:", compatibility_check["status"])
print("query types:", len(query_entries))
print("query instances:", query_instances)
print("target links:", target_links)
print("image embeddings:", tuple(image_embeddings.shape), "ids:", len(image_ids))
print("text embeddings:", tuple(text_embeddings.shape), "prompts:", len(text_ids))

# 1. Introduction

# 2. Related work

# 3. Method

[a detailed, formal overview of the solution you developed.
This must include a mathematical description of your architecture, the forward pass, and
the loss functions governing the training process (if applicable). Clearly highlight your
original contributions and adequately cite relevant literature.]

# 4. Experiments and results

[a rigorous description of the training and evaluation strategy. Exten
sively motivate your methodological choices, including network capacity, optimizer selec
tion, hyperparameter tuning, and data sampling strategies]

i suggest to put here also: [an extensive presentation of your findings. You must report stan
dard retrieval metrics (Recall@K). Organize your scores in comparative tables, and in
clude charts depicting learning curves (if applicable), qualitative retrieval examples (suc
cesses and failure cases), and any other visual representations that aid in understanding the
model’s behavior]

## Baseline Formula

For each query instance, the baseline builds:

```text
q = normalize(v_ref + alpha * sum(positive_text) - beta * sum(negative_text))
```

where:

- `v_ref` is the CLIP image embedding of the reference image;
- `positive_text` contains CLIP text embeddings for attributes such as `+Smiling`;
- `negative_text` contains CLIP text embeddings for attributes such as `-Eyeglasses`;
- retrieval uses dot product because all embeddings are L2-normalized.

## Run the Best Baseline from the Sweep

The best setting from the small alpha/beta sweep was `alpha=2.0`, `beta=1.0`.
This cell saves only `metrics.json` to keep experiment folders small.

In [ ]:
result = run_baseline(
    query_json=QUERY_JSON,
    image_embedding_dir=IMAGE_EMBEDDING_DIR,
    text_embedding_dir=TEXT_EMBEDDING_DIR,
    output_dir=OUTPUT_DIR,
    alpha=2.0,
    beta=1.0,
    save_predictions=False,
    save_run_config=False,
)

pd.DataFrame([result["metrics"]])

## Alpha/Beta Sweep Summary

The sweep below compares different text-direction weights. Higher `alpha` means stronger positive attribute push; higher `beta` means stronger negative attribute push.

In [ ]:
summary = pd.read_csv(SUMMARY_CSV)
summary.sort_values("recall@10", ascending=False).reset_index(drop=True)

## Interpretation

The baseline is intentionally simple: it does not learn a transformation and does not use visual attribute directions.
The best run in the current sweep, `alpha=2.0` and `beta=1.0`, improves recall@10 over the default `alpha=1.0`, `beta=1.0`.

This suggests that, on this subset, a stronger positive text direction helps. The next natural improvement is to replace raw text directions with visual attribute directions, or to add a small gating mechanism that controls how strongly each attribute modifies the reference image.

These numbers are useful for development, but they are subset results. Final assignment numbers should be reported on the official full CelebA test split when the full test embeddings are available.

## Visual-Direction Retrieval

After the CLIP text-arithmetic baseline, we tested a second retrieval variant based on visual attribute directions. For each CelebA attribute, the direction was computed from the train split as:

```text
direction(attribute) = normalize(mean(images where attribute = +1) - mean(images where attribute = -1))
```

The retrieval query then used the reference image plus signed visual directions:

```text
q = normalize(alpha * reference_image + beta_pos * sum(positive directions) - beta_neg * sum(negative directions))
```

This was implemented separately from the baseline in `src/retrieval/visual_direction_retrieval.py`, so the baseline results remain a fixed reference point.


## Visual-Direction Sweep

We evaluated a small fixed-weight grid using the same test subset, query file, and metrics as the baseline:

```text
alpha:    0.5, 1.0, 2.0
beta_pos: 0.5, 1.0, 2.0
beta_neg: 0.5, 1.0, 2.0
```

The sweep was run with `scripts/run_visual_direction_retrieval.py`, and the sorted results were saved to `outputs/visual_direction_run/summary.csv`.


In [ ]:
visual_summary_path = PROJECT_ROOT / "outputs/visual_direction_run/summary.csv"
visual_summary = pd.read_csv(visual_summary_path)
visual_summary.head(5)[[
    "alpha", "beta_pos", "beta_neg",
    "recall@1", "precision@1",
    "recall@5", "precision@5",
    "recall@10", "precision@10",
]]


## Baseline vs Visual Directions

The best visual-direction setting was compared against the best baseline setting. The visual-direction run was close, but it did not improve over the CLIP text-arithmetic baseline.


In [ ]:
best_baseline = summary.sort_values("recall@10", ascending=False).iloc[0]
best_visual = visual_summary.sort_values("recall@10", ascending=False).iloc[0]

comparison = pd.DataFrame([
    {
        "method": "CLIP text arithmetic baseline",
        "alpha": best_baseline["alpha"],
        "beta": best_baseline["beta"],
        "beta_pos": None,
        "beta_neg": None,
        "recall@1": best_baseline["recall@1"],
        "recall@5": best_baseline["recall@5"],
        "recall@10": best_baseline["recall@10"],
        "precision@10": best_baseline["precision@10"],
    },
    {
        "method": "visual directions",
        "alpha": best_visual["alpha"],
        "beta": None,
        "beta_pos": best_visual["beta_pos"],
        "beta_neg": best_visual["beta_neg"],
        "recall@1": best_visual["recall@1"],
        "recall@5": best_visual["recall@5"],
        "recall@10": best_visual["recall@10"],
        "precision@10": best_visual["precision@10"],
    },
])
comparison


## Visual-Direction Takeaway

The best visual-direction setting was `alpha=2.0`, `beta_pos=1.0`, `beta_neg=0.5`, with `recall@1=0.0506`, `recall@5=0.1463`, and `recall@10=0.2183`. The best baseline setting was `alpha=2.0`, `beta=1.0`, with `recall@1=0.0561`, `recall@5=0.1500`, and `recall@10=0.2262`. Therefore, simple fixed visual directions were competitive but slightly worse than text directions. This suggests that raw visual directions alone may be noisy or too global, and the next improvement should probably combine text and visual directions or weight attributes based on difficulty/reliability. The strongest visual-direction runs consistently favored a smaller negative-direction weight, with the best setting using `beta_neg=0.5`; this suggests that negative visual directions are less trustworthy in this setup and may be more destructive than positive directions when they are applied too strongly.


## Hybrid Text + Visual Fusion

Since visual directions alone may be noisy and may contain correlated CelebA attribute information, we tested a query-level hybrid that keeps the CLIP text direction as the main semantic signal and adds visual directions as a dataset-specific correction. The query is:

```text
q = normalize(
    alpha * reference_image_embedding
    + beta_text * text_delta
    + beta_visual_pos * visual_pos_delta
    - beta_visual_neg * visual_neg_delta
)
```

where `text_delta = sum(text_embeddings[pos_attrs]) - sum(text_embeddings[neg_attrs])`, `visual_pos_delta = sum(visual_directions[pos_attrs])`, and `visual_neg_delta = sum(visual_directions[neg_attrs])`. This was implemented in `src/retrieval/hybrid_fusion_retrieval.py` and run with `scripts/run_hybrid_fusion_retrieval.py`.


In [ ]:
hybrid_summary_path = PROJECT_ROOT / "outputs/hybrid_fusion_run/summary.csv"
hybrid_summary = pd.read_csv(hybrid_summary_path)
best_hybrid = hybrid_summary.sort_values("recall@10", ascending=False).iloc[0]

hybrid_comparison = pd.DataFrame([
    {
        "method": "CLIP text arithmetic baseline",
        "alpha": best_baseline["alpha"],
        "beta_text": best_baseline["beta"],
        "beta_visual_pos": None,
        "beta_visual_neg": None,
        "recall@1": best_baseline["recall@1"],
        "recall@5": best_baseline["recall@5"],
        "recall@10": best_baseline["recall@10"],
        "precision@10": best_baseline["precision@10"],
    },
    {
        "method": "visual directions",
        "alpha": best_visual["alpha"],
        "beta_text": None,
        "beta_visual_pos": best_visual["beta_pos"],
        "beta_visual_neg": best_visual["beta_neg"],
        "recall@1": best_visual["recall@1"],
        "recall@5": best_visual["recall@5"],
        "recall@10": best_visual["recall@10"],
        "precision@10": best_visual["precision@10"],
    },
    {
        "method": "hybrid text + visual fusion",
        "alpha": best_hybrid["alpha"],
        "beta_text": best_hybrid["beta_text"],
        "beta_visual_pos": best_hybrid["beta_visual_pos"],
        "beta_visual_neg": best_hybrid["beta_visual_neg"],
        "recall@1": best_hybrid["recall@1"],
        "recall@5": best_hybrid["recall@5"],
        "recall@10": best_hybrid["recall@10"],
        "precision@10": best_hybrid["precision@10"],
    },
])

hybrid_comparison


## Hybrid Takeaway

The best hybrid setting was `alpha=1.0`, `beta_text=2.0`, `beta_visual_pos=0.5`, and `beta_visual_neg=0.25`, with `recall@1=0.0671`, `recall@5=0.1927`, `recall@10=0.2860`, and `precision@10=0.0413`. This improves over both the best text-only baseline (`recall@10=0.2262`) and the best visual-direction-only run (`recall@10=0.2183`). The smaller negative visual weight again supports the hypothesis that negative visual directions are less reliable than positive ones when used too strongly.

This parameter pattern is also informative. The text-only baseline needed a larger reference weight (`alpha=2.0`), while the hybrid works best with a smaller reference weight (`alpha=1.0`) and a stronger modification signal (`beta_text=2.0`). This suggests that, once the attribute modification is supported by both CLIP text embeddings and CelebA visual directions, the query can move farther away from the original reference image and rely more on the requested edit.

A natural next step is therefore a finer grid around this region:

```text
alpha:           0.5, 0.75, 1.0, 1.25, 1.5
beta_text:       1.5, 2.0, 2.5, 3.0
beta_visual_pos: 0.25, 0.5, 0.75, 1.0
beta_visual_neg: 0.0, 0.1, 0.25, 0.4, 0.5
```

The `beta_visual_neg=0.0` case is especially important: if negative visual directions are very noisy, the best hybrid may use visual directions only for positive attributes and leave negative edits to the CLIP text direction.



## Prompt-Ensemble Hybrid Check

As a small follow-up, we replaced the single CLIP attribute prompt in the hybrid method with an average of multiple positive prompt templates per attribute. The retrieval formula stays the same; only the text embedding used inside `text_delta` changes.


In [ ]:
prompt_ensemble_summary_path = PROJECT_ROOT / "outputs/hybrid_prompt_ensemble_run/summary.csv"
prompt_ensemble_summary = pd.read_csv(prompt_ensemble_summary_path)
best_prompt_ensemble = prompt_ensemble_summary.sort_values("recall@10", ascending=False).iloc[0]

pd.DataFrame([
    {
        "method": "single-prompt hybrid",
        "recall@10": best_hybrid["recall@10"],
        "precision@10": best_hybrid["precision@10"],
    },
    {
        "method": "prompt-ensemble hybrid",
        "recall@10": best_prompt_ensemble["recall@10"],
        "precision@10": best_prompt_ensemble["precision@10"],
    },
])


## Prompt-Ensemble Takeaway

Prompt ensembling did not improve the main retrieval score in this experiment: the single-prompt hybrid reached `recall@10=0.2860`, while the prompt-ensemble hybrid reached `recall@10=0.2811`. This suggests that averaging several text prompts mostly smooths the text signal, but does not add a useful correction beyond what the single prompt and CelebA visual directions already provide.

The prompt-ensemble run has a slightly higher `precision@10` (`0.0416` vs `0.0413`), but the difference is very small, so it is better treated as a side observation rather than the main direction. The strongest method remains the hybrid with the original CLIP text prompt plus visual directions: CLIP text provides the semantic edit, and the CelebA visual directions provide the dataset-specific adjustment.


# 5. Discussion and conclusion